# Classical Models and Training
Predict first, run the calculations and explain the differences. Data are generated locally with fixed seeds; reported scores describe these teaching examples, not general performance guarantees.


In [ ]:
%pip -q install "numpy>=2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


## Independent gradient descent and a least-squares reference
The model is y=wx+b. Update both parameters using the old gradient.

**Predict:** The first simultaneous update at learning rate 0.1 gives w=13/15 and b=3/5. Calculate its MSE before running.


In [ ]:
x=np.array([0.,1.,2.]); y=np.array([1.,3.,5.])
w=b=0.
losses=[]
for step in range(1000):
    r=w*x+b-y
    losses.append(np.mean(r*r))
    dw=2*np.mean(r*x); db=2*np.mean(r)
    w,b=w-.1*dw,b-.1*db
reference=np.linalg.lstsq(np.column_stack([x,np.ones(3)]),y,rcond=None)[0]
assert np.allclose([w,b],reference,atol=1e-7)
assert np.isclose(losses[0],35/3)
print("Fitted parameters",w,b,"reference",reference)


## A nonlinear representation for XOR
This hand calculation illustrates expressive capacity; it is not a trained network.

**Predict:** Tabulate both hidden activations at all Boolean corners. Keep this specified representation distinct from the subsequently fitted MLP.


In [ ]:
corners=np.array([[0,0],[0,1],[1,0],[1,1]])
s=corners.sum(axis=1)
h1=np.maximum(0,s);h2=np.maximum(0,s-1)
assert np.allclose(h1-2*h2,[0,1,1,0])
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons
Xc,yc=make_moons(n_samples=220,noise=.18,random_state=2026)
Xct,Xce,yct,yce=train_test_split(Xc,yc,stratify=yc,test_size=.25,random_state=42)
network=make_pipeline(StandardScaler(),MLPClassifier(hidden_layer_sizes=(8,),solver="lbfgs",max_iter=3000,random_state=2026))
network.fit(Xct,yct)
print("Predeclared neural model test accuracy:",network.score(Xce,yce))


## Model selection stays inside cross-validation
Feature expansion and scaling are inside the pipeline. The test set is evaluated only after the grid search. Penalty conventions follow scikit-learn Ridge; its alpha is not numerically identical to a lambda multiplying a mean loss without a scale conversion.


In [ ]:
rng=np.random.default_rng(2026)
X=rng.uniform(-2,2,(160,1)); y=np.sin(2*X[:,0])+rng.normal(0,.2,160)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42)
pipe=make_pipeline(PolynomialFeatures(include_bias=False),StandardScaler(),Ridge())
cv=KFold(n_splits=4,shuffle=True,random_state=42)
search=GridSearchCV(pipe,{"polynomialfeatures__degree":[1,3,8],"ridge__alpha":[.01,1,100]},scoring="neg_mean_squared_error",cv=cv,return_train_score=True)
search.fit(Xtr,ytr)
print("Selected with CV:",search.best_params_)
print("Final held-out MSE:",mean_squared_error(yte,search.predict(Xte)))
assert np.allclose(search.best_estimator_[1].mean_,search.best_estimator_[0].transform(Xtr).mean(axis=0))


## Exercises
1. Increase the learning rate in the gradient calculation and inspect the loss rather than assuming convergence.
2. Compare train and validation errors across polynomial degrees.
3. Explain why the test score must not guide another round of degree selection.


## Plot model selection without tuning on test data
Compare training and cross-validation MSE at each penalty. Explain why the smallest training error does not automatically select the procedure. The grey curve in the browser uses a known noiseless teaching reference; the curves here use actual held-out CV folds.


In [ ]:
import matplotlib.pyplot as plt
results = search.cv_results_
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, penalty in zip(axes, [.01, 1, 100]):
    chosen = [i for i,p in enumerate(results["params"]) if p["ridge__alpha"] == penalty]
    degree = [results["params"][i]["polynomialfeatures__degree"] for i in chosen]
    ax.plot(degree, -results["mean_train_score"][chosen], "o-", label="Training")
    ax.plot(degree, -results["mean_test_score"][chosen], "o-", label="CV validation")
    ax.set(xlabel="Polynomial degree", title=f"Ridge alpha={penalty}")
axes[0].set_ylabel("MSE"); axes[0].legend(); fig.tight_layout(); plt.show()


## Check one backpropagation calculation independently
Use the active-ReLU example from Lesson 11.2. Predict d(loss)/du=−1 and d(loss)/dv=−1/2. Central finite differences check the calculation away from the ReLU kink; they are a diagnostic, not the training algorithm.


In [ ]:
def tiny_loss(u, v):
    h = max(0., u)  # x=1, hidden bias=0
    logit = v*h - 2.
    return np.logaddexp(0., logit) - logit  # stable BCE for y=1
eps = 1e-5
du = (tiny_loss(1+eps,2)-tiny_loss(1-eps,2))/(2*eps)
dv = (tiny_loss(1,2+eps)-tiny_loss(1,2-eps))/(2*eps)
assert np.allclose([du,dv],[-1.,-.5],atol=1e-8)
first_loss = np.mean(((13/15)*np.array([0.,1.,2.])+3/5-np.array([1.,3.,5.]))**2)
assert np.isclose(first_loss,433/135)
print("Independent derivatives:", du, dv, "first regression-step MSE:", first_loss)
